# 03 — 端到端管线

完整跑通 index + query，对比纯向量 vs hybrid 检索效果。

In [ ]:
import sys
sys.path.insert(0, '..')
import logging
logging.basicConfig(level=logging.WARNING, format='%(levelname)s:%(message)s')

# 按需设置 HF 镜像
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

from src.pipeline import RAGPipeline
from src.embedding import EmbeddingModel
from src.retrieval import Retriever
from src.vector_store import VectorStore

In [ ]:
# 初始化
p = RAGPipeline()
p.embed_model.model_name = 'BAAI/bge-small-zh-v1.5'

# 索引
n = p.index()
print(f'Indexed {n} chunks')

## 检索对比：纯向量 vs Hybrid

In [ ]:
test_questions = [
    '什么是注意力机制',
    'Transformer 和 RNN 有什么区别',
    'MHA 是什么',
]

embed = p.embed_model

for q in test_questions:
    print(f'\n=== Q: {q} ===')
    qvec = embed.embed_text(q)
    
    # 纯向量检索
    dense_retriever = Retriever(vector_store=p.vector_store, mode='dense')
    dense_results = dense_retriever.dense_search(qvec, top_k=3)
    print(f'  Dense: {len(dense_results)} results')
    for r in dense_results:
        print(f'    [{r["metadata"].get("source", "?")}] score={r["score"]:.4f}')
    
    # Hybrid 检索
    hybrid_retriever = Retriever(vector_store=p.vector_store, bm25_index=p.bm25_index, mode='hybrid')
    hybrid_results = hybrid_retriever.hybrid_search(qvec, q, top_k=3)
    print(f'  Hybrid: {len(hybrid_results)} results')
    for r in hybrid_results:
        print(f'    [{r.get("metadata", {}).get("source", "?")}] score={r["score"]:.4f}')

## 端到端查询

注：需要先在 `.env` 中配置 API Key。

In [ ]:
# result = p.query('什么是注意力机制?')
# print(f'A: {result["answer"]}')
# print(f'Sources: {len(result["sources"])} chunks')